# 04_modelo — Previsão de Churn

Treina, avalia e aplica o modelo de previsão de churn com horizonte de 3 meses.

**Input:** `data/processed/df_model.parquet`
**Output:** `data/processed/scores_2026.parquet`, modelo salvo em `data/processed/modelo_churn.pkl`

---
## Seção 1 — Setup

In [1]:

import sys, os, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

PROJECT_ROOT = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / "src").exists())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src.config import PROCESSED_DIR, VERDE

print("Setup OK")
print("Working directory:", os.getcwd())

Setup OK
Working directory: g:\Meu Drive\central_eto


In [2]:
df_treino = pd.read_parquet(PROCESSED_DIR / "df_model_treino.parquet")
df_teste  = pd.read_parquet(PROCESSED_DIR / "df_model_teste.parquet")

print(f"Treino : {df_treino.shape}  |  churn: {df_treino['churn'].mean():.1%}")
print(f"Teste  : {df_teste.shape}   |  churn: {df_teste['churn'].mean():.1%}")

Treino : (610, 25)  |  churn: 36.2%
Teste  : (724, 21)   |  churn: 49.0%


In [3]:
print("=== Treino ===")
print(df_treino["churn"].value_counts().rename({0: "Ativo", 1: "Churnou"}))
print(f"Missing: {df_treino.isna().sum().sum()}")

print("\n=== Teste ===")
print(df_teste["churn"].value_counts().rename({0: "Ativo", 1: "Churnou"}))
print(f"Missing: {df_teste.isna().sum().sum()}")

=== Treino ===
churn
Ativo      389
Churnou    221
Name: count, dtype: int64
Missing: 166

=== Teste ===
churn
Ativo      369
Churnou    355
Name: count, dtype: int64
Missing: 0


In [ ]:
IDENTIFICADORES = ["CLIENTE", "NOME"]
TARGET = "churn"

MAPA_CATEGORIA = {"baixo": 0, "medio": 1, "alto": 2, "premium": 3}

def preparar_xy(df):
    X = df.drop(columns=IDENTIFICADORES + [TARGET]).copy()
    for col in ["categoria_pedido", "categoria_cliente"]:
        if col in X.columns:
            X[col] = X[col].astype("object").map(MAPA_CATEGORIA)
    y = df[TARGET]
    return X, y

X_train, y_train = preparar_xy(df_treino)
X_test,  y_test  = preparar_xy(df_teste)

print(f"X_train: {X_train.shape}  |  churn: {y_train.mean():.1%}")
print(f"X_test : {X_test.shape}   |  churn: {y_test.mean():.1%}")

---
## Split temporal

Treino e teste são snapshots em pontos diferentes no tempo — sem split aleatório.

| Conjunto | CUTOFF | Features | Outcome |
|---|---|---|---|
| Treino | dez/2024 | jan/23–dez/24 | churn em 2025 (janela por categoria) |
| Teste  | dez/2025 | jan/23–dez/25 | churn em 2026 (janela por categoria, travada em mai/26) |

In [5]:
# X_train, y_train e X_test, y_test já foram definidos na célula anterior
# com base nos dois parquets de treino e teste

print(f"Treino : {X_train.shape[0]} clientes  |  churn: {y_train.mean():.1%}")
print(f"Teste  : {X_test.shape[0]} clientes   |  churn: {y_test.mean():.1%}")

Treino : 610 clientes  |  churn: 36.2%
Teste  : 724 clientes   |  churn: 49.0%


---
## Seção 2 — EDA da Modelagem

In [ ]:
from sklearn.feature_selection import mutual_info_classif

X_num = X_train.select_dtypes(include="number")
pearson = X_num.corrwith(y_train).abs().sort_values(ascending=False).rename("pearson")

mi_scores = mutual_info_classif(X_num.fillna(0), y_train, random_state=42)
mi = pd.Series(mi_scores, index=X_num.columns).sort_values(ascending=False).rename("mutual_info")

corr_target = pd.concat([pearson, mi], axis=1).sort_values("pearson", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

corr_target["pearson"].sort_values().plot(
    kind="barh", ax=axes[0], color=VERDE[5], edgecolor="white"
)
axes[0].set_title("Correlação Pearson com target", fontsize=13)
axes[0].set_xlabel("|ρ|")

corr_target["mutual_info"].sort_values().plot(
    kind="barh", ax=axes[1], color=VERDE[4], edgecolor="white"
)
axes[1].set_title("Mutual Information com target", fontsize=13)
axes[1].set_xlabel("MI score")

plt.suptitle("Relevância das features para churn", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print(corr_target.round(3).to_string())

In [ ]:
# === Análise exploratória: sinal com o target ===
# Identifica features com Pearson E MI abaixo do threshold.
# ATENÇÃO: isso é uma hipótese de remoção, não uma decisão final.
# Features fracas individualmente podem contribuir via interações no XGBoost.
# A decisão definitiva de remoção virá do SHAP (Seção 9).

PEARSON_MIN = 0.10
MI_MIN      = 0.05

baixo_sinal = corr_target[
    (corr_target["pearson"]     < PEARSON_MIN) &
    (corr_target["mutual_info"] < MI_MIN)
].sort_values("pearson")

print(f"Features com Pearson < {PEARSON_MIN} E MI < {MI_MIN} (hipóteses de remoção):\n")
print(f"{'Feature':<25} {'Pearson':>8}  {'MI':>8}")
print("-" * 45)
for feat, row in baixo_sinal.iterrows():
    print(f"{feat:<25} {row['pearson']:>8.3f}  {row['mutual_info']:>8.3f}")

HIPOTESES_BAIXO_SINAL = baixo_sinal.index.tolist()
print(f"\n→ {len(HIPOTESES_BAIXO_SINAL)} hipóteses: {HIPOTESES_BAIXO_SINAL}")
print("→ Confirmar ou descartar via SHAP na Seção 9.")

In [7]:
top_features = corr_target["pearson"].sort_values(ascending=False).head(10).index.tolist()

df_plot = X_train[top_features].copy()
df_plot["churn"] = y_train.values

n_cols = 3
n_rows = -(-len(top_features) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    grupos = [df_plot.loc[df_plot["churn"] == 0, feat], df_plot.loc[df_plot["churn"] == 1, feat]]
    axes[i].boxplot(grupos, labels=["Ativo", "Churnou"], patch_artist=True,
                    boxprops=dict(facecolor=VERDE[3], alpha=0.7),
                    medianprops=dict(color=VERDE[7], linewidth=2))
    axes[i].set_title(feat, fontsize=11)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribuição das top features por classe", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

NameError: name 'corr_target' is not defined

In [ ]:
import seaborn as sns

corr_matrix = X_train.select_dtypes(include="number").corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap=sns.light_palette(VERDE[5], as_cmap=True),
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    ax=ax,
    vmin=-1, vmax=1,
    center=0,
)
ax.set_title("Correlação entre features", fontsize=14)
plt.tight_layout()
plt.show()

alta_corr = (
    corr_matrix.abs()
    .where(np.tril(np.ones(corr_matrix.shape), k=-1).astype(bool))
    .stack()
    .reset_index()
    .rename(columns={"level_0": "feat_a", "level_1": "feat_b", 0: "corr"})
    .query("corr > 0.7")
    .sort_values("corr", ascending=False)
)

if alta_corr.empty:
    print("Nenhum par com correlação > 0.7")
else:
    print("Pares com correlação > 0.7:")
    print(alta_corr.to_string(index=False))

---
## Seção 3 — Análise Exploratória de Features

Identifica **hipóteses** de remoção por dois critérios independentes:
1. **Alta correlação entre features** → uma delas é redundante
2. *(já feito na Seção 2)* Baixo sinal com o target (Pearson + MI)

Estas listas **não são filtros definitivos** — features com sinal fraco individualmente podem contribuir via interações no XGBoost. A decisão final de remoção virá do SHAP (Seção 9).

**Exceção:** `razao_atividade` é transformação linear de `n_meses_ativos` (divisão por constante) — remoção garantida independente do modelo.

In [ ]:
CORR_THRESHOLD = 0.80

# Score combinado normalizado: Pearson + MI, cada um escalado para [0, 1]
score_target = corr_target.assign(
    p_norm = lambda d: d["pearson"]     / d["pearson"].max(),
    m_norm = lambda d: d["mutual_info"] / d["mutual_info"].max(),
    score  = lambda d: d["p_norm"] + d["m_norm"],
)["score"]

# Correlação entre features (fillna=0 apenas para o cálculo da matriz)
X_num   = X_train.select_dtypes(include="number")
corr_ff = X_num.fillna(0).corr().abs()

pares = (
    corr_ff
    .where(np.tril(np.ones(corr_ff.shape), k=-1).astype(bool))
    .stack()
    .reset_index()
    .rename(columns={"level_0": "feat_a", "level_1": "feat_b", 0: "corr_ff"})
    .query("corr_ff > @CORR_THRESHOLD")
    .sort_values("corr_ff", ascending=False)
    .reset_index(drop=True)
)

rows = []
for _, r in pares.iterrows():
    a, b   = r["feat_a"], r["feat_b"]
    sa, sb = score_target.get(a, 0), score_target.get(b, 0)
    remover, manter = (a, b) if sa <= sb else (b, a)
    rows.append({
        "remover":       remover,
        "manter":        manter,
        "corr_entre_si": round(r["corr_ff"], 3),
        "score_remover": round(score_target.get(remover, 0), 3),
        "score_manter":  round(score_target.get(manter,  0), 3),
    })

df_hipoteses_corr = (
    pd.DataFrame(rows)
    .drop_duplicates(subset="remover")
    .reset_index(drop=True)
)

print(f"Hipóteses por alta correlação entre features (threshold: {CORR_THRESHOLD}):\n")
print(f"{'REMOVER':<25} {'MANTER':<25} {'corr':>6}  {'sc_rem':>6}  {'sc_man':>6}")
print("=" * 65)
for _, r in df_hipoteses_corr.iterrows():
    print(f"{r['remover']:<25} {r['manter']:<25} {r['corr_entre_si']:>6.3f}  {r['score_remover']:>6.3f}  {r['score_manter']:>6.3f}")

HIPOTESES_REDUNDANCIA = df_hipoteses_corr["remover"].tolist()
print(f"\n→ {len(HIPOTESES_REDUNDANCIA)} hipóteses: {HIPOTESES_REDUNDANCIA}")
print("→ Confirmar ou descartar via SHAP na Seção 9.")
print("→ Exceção: 'razao_atividade' será removida agora (transformação linear de n_meses_ativos).")

# Remoção garantida — transformação linear, zero informação nova
REMOVER_AGORA = ["razao_atividade"]
X_train_clean = X_train.drop(columns=REMOVER_AGORA, errors="ignore")
X_test_clean  = X_test.drop(columns=REMOVER_AGORA, errors="ignore")
print(f"\nX_train após remoção garantida: {X_train_clean.shape}")

---
## Seção 4 — Distribuição de Classes

Verificar se o imbalanceamento exige intervenção antes da modelagem.

| Situação | Decisão |
|---|---|
| Imbalanceamento severo (< 10% positivos) | SMOTE ou `scale_pos_weight` obrigatório |
| Imbalanceamento moderado (10–30%) | `scale_pos_weight` recomendado |
| Balanceamento razoável (> 30%) | Sem intervenção — threshold resolve |

`scale_pos_weight` não será aplicado como pré-processamento fixo: entra como **hiperparâmetro no Optuna** (Seção 8) para que a validação cruzada decida se ajuda.

O shift de distribuição entre treino (36%) e teste (49%) é real mas não é imbalanceamento — é reflexo de um período com mais clientes novos e voláteis. Tratado por **calibração de threshold no conjunto de teste** (Seção 7).

In [ ]:
n_pos   = y_train.sum()
n_neg   = (y_train == 0).sum()
pct_pos = y_train.mean()

print(f"Treino — Ativo (0): {n_neg}  |  Churnou (1): {n_pos}  |  Taxa churn: {pct_pos:.1%}")
print(f"scale_pos_weight referência (n_neg / n_pos): {n_neg / n_pos:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Treino
axes[0].bar(["Ativo", "Churnou"], [n_neg, n_pos],
            color=[VERDE[3], VERDE[6]], edgecolor="white")
axes[0].set_title(f"Treino — {pct_pos:.1%} churn", fontsize=12)
axes[0].set_ylabel("Clientes")
for i, v in enumerate([n_neg, n_pos]):
    axes[0].text(i, v + 5, str(v), ha="center", fontsize=11)

# Teste
n_pos_t = y_test.sum()
n_neg_t = (y_test == 0).sum()
axes[1].bar(["Ativo", "Churnou"], [n_neg_t, n_pos_t],
            color=[VERDE[3], VERDE[6]], edgecolor="white")
axes[1].set_title(f"Teste — {y_test.mean():.1%} churn", fontsize=12)
axes[1].set_ylabel("Clientes")
for i, v in enumerate([n_neg_t, n_pos_t]):
    axes[1].text(i, v + 5, str(v), ha="center", fontsize=11)

plt.suptitle("Distribuição de classes — treino vs teste", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

if pct_pos >= 0.30:
    print("\n✓ Balanceamento razoável (> 30%) — sem intervenção obrigatória.")
    print("  scale_pos_weight tratado como hiperparâmetro no Optuna (Seção 8).")
elif pct_pos >= 0.10:
    print("\n⚠ Imbalanceamento moderado — considerar scale_pos_weight.")
else:
    print("\n✗ Imbalanceamento severo — intervenção necessária (SMOTE ou scale_pos_weight).")

---
## Seção 5 — Baseline

Régua de comparação para o XGBoost. Todo modelo precisa bater esses números.

| Subseção | Modelo | Objetivo |
|---|---|---|
| 5.1 | DummyClassifier | Chão absoluto — AUC-ROC ~0.50 por definição |
| 5.2 | VIF | Verifica se a logística é viável (VIF > 10 → feature instável no modelo linear) |
| 5.3 | Logistic Regression | Baseline interpretável — se o XGBoost não bater por ~3–5 pts de AUC, o problema é linear |

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# --- 5.1 DummyClassifier ---
dummy = DummyClassifier(strategy="stratified", random_state=42)
dummy.fit(X_train_clean, y_train)

y_prob_dummy  = dummy.predict_proba(X_test_clean)[:, 1]
auc_roc_dummy = roc_auc_score(y_test, y_prob_dummy)
auc_pr_dummy  = average_precision_score(y_test, y_prob_dummy)

print("5.1 — DummyClassifier (stratified)")
print(f"  AUC-ROC : {auc_roc_dummy:.4f}  (esperado ~0.50)")
print(f"  AUC-PR  : {auc_pr_dummy:.4f}  (esperado ~{y_test.mean():.2f} = prevalência do teste)")

In [ ]:
from sklearn.linear_model import LinearRegression

# --- 5.2 VIF — implementado com sklearn, sem statsmodels ---
# VIF_i = 1 / (1 - R²), onde R² é obtido regredindo a feature i contra todas as outras.
# Só relevante para o modelo linear (Seção 5.3) — XGBoost não é afetado por multicolinearidade.

def calc_vif(X: pd.DataFrame) -> pd.DataFrame:
    rows = []
    cols = X.columns.tolist()
    for col in cols:
        X_other = X.drop(columns=[col]).values
        y_col   = X[col].values
        r2 = LinearRegression().fit(X_other, y_col).score(X_other, y_col)
        rows.append({"feature": col, "VIF": np.inf if r2 >= 1.0 else 1 / (1 - r2)})
    return pd.DataFrame(rows).sort_values("VIF", ascending=False).reset_index(drop=True)

X_vif = X_train_clean.select_dtypes(include="number").fillna(0)
vif   = calc_vif(X_vif)

print("5.2 — VIF (Variance Inflation Factor)")
print(f"{'Feature':<25} {'VIF':>8}")
print("-" * 35)
for _, r in vif.iterrows():
    flag = "  ← excluir da logística" if r["VIF"] > 10 else ""
    print(f"{r['feature']:<25} {r['VIF']:>8.2f}{flag}")

EXCLUIR_LOGISTICA = vif.loc[vif["VIF"] > 10, "feature"].tolist()
print(f"\nFeatures excluídas da logística (VIF > 10): {EXCLUIR_LOGISTICA}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# --- 5.3 Logistic Regression ---
X_train_lr = X_train_clean.drop(columns=EXCLUIR_LOGISTICA, errors="ignore").fillna(0)
X_test_lr  = X_test_clean.drop(columns=EXCLUIR_LOGISTICA, errors="ignore").fillna(0)

pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
pipe_lr.fit(X_train_lr, y_train)

y_prob_lr  = pipe_lr.predict_proba(X_test_lr)[:, 1]
auc_roc_lr = roc_auc_score(y_test, y_prob_lr)
auc_pr_lr  = average_precision_score(y_test, y_prob_lr)

print("5.3 — Logistic Regression")
print(f"  Features usadas : {X_train_lr.shape[1]}")
print(f"  AUC-ROC (teste) : {auc_roc_lr:.4f}")
print(f"  AUC-PR  (teste) : {auc_pr_lr:.4f}")

resultados = pd.DataFrame([
    {"Modelo": "DummyClassifier",    "AUC-ROC": auc_roc_dummy, "AUC-PR": auc_pr_dummy},
    {"Modelo": "LogisticRegression", "AUC-ROC": auc_roc_lr,    "AUC-PR": auc_pr_lr},
])
print("\n=== Comparação de modelos ===")
print(resultados.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
coefs = pd.Series(
    pipe_lr.named_steps["model"].coef_[0],
    index=X_train_lr.columns,
).sort_values()

cores = [VERDE[6] if v > 0 else VERDE[2] for v in coefs]

fig, ax = plt.subplots(figsize=(8, max(4, len(coefs) * 0.4)))
coefs.plot(kind="barh", ax=ax, color=cores, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Coeficientes da Regressão Logística (após StandardScaler)", fontsize=12)
ax.set_xlabel("Coeficiente (positivo → aumenta P(churn))")
plt.tight_layout()
plt.show()

---
## Seção 6 — Comparação de Modelos Tree-Based

Treina três modelos com parâmetros razoáveis mas sem tuning fino.  
O vencedor desta seção vai para o Optuna na Seção 8.

| Modelo | Característica |
|---|---|
| XGBoost | Boosting — geralmente melhor AUC, mais lento |
| LightGBM | Boosting — mais rápido, similar ao XGBoost |
| Random Forest | Bagging — mais estável, geralmente AUC menor que boosting |

`scale_pos_weight` entra como hiperparâmetro no Optuna do modelo vencedor — não aqui.

In [ ]:
from xgboost import XGBClassifier

X_train_xgb = X_train_clean.fillna(0)
X_test_xgb  = X_test_clean.fillna(0)

xgb = XGBClassifier(
    n_estimators     = 300,
    learning_rate    = 0.05,
    max_depth        = 4,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    eval_metric      = "aucpr",
    early_stopping_rounds = 20,
    random_state     = 42,
    verbosity        = 0,
)
xgb.fit(
    X_train_xgb, y_train,
    eval_set=[(X_test_xgb, y_test)],
    verbose=False,
)

y_prob_xgb  = xgb.predict_proba(X_test_xgb)[:, 1]
auc_roc_xgb = roc_auc_score(y_test, y_prob_xgb)
auc_pr_xgb  = average_precision_score(y_test, y_prob_xgb)

print(f"XGBoost (default)  —  árvores usadas: {xgb.best_iteration + 1} / {xgb.n_estimators}")
print(f"  AUC-ROC (teste) : {auc_roc_xgb:.4f}")
print(f"  AUC-PR  (teste) : {auc_pr_xgb:.4f}")

# Atualizar tabela de comparação
resultados = pd.concat([
    resultados,
    pd.DataFrame([{"Modelo": "XGBoost (default)", "AUC-ROC": auc_roc_xgb, "AUC-PR": auc_pr_xgb}])
], ignore_index=True)

print("\n=== Comparação de modelos ===")
print(resultados.to_string(index=False, float_format="{:.4f}".format))

# Diagnóstico rápido
delta_lr  = auc_roc_xgb - auc_roc_lr
delta_dum = auc_roc_xgb - auc_roc_dummy
print(f"\nGanho vs Dummy     : +{delta_dum:.4f}")
print(f"Ganho vs Logística : {delta_lr:+.4f}  {'✓ não-linear capturado' if delta_lr > 0.03 else '⚠ ganho marginal — revisar features antes de tunar'}")

In [ ]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

lgbm = LGBMClassifier(
    n_estimators     = 300,
    learning_rate    = 0.05,
    max_depth        = 4,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    random_state     = 42,
    verbose          = -1,
)
lgbm.fit(
    X_train_xgb, y_train,
    eval_set=[(X_test_xgb, y_test)],
    callbacks=[early_stopping(20, verbose=False), log_evaluation(-1)],
)

y_prob_lgbm  = lgbm.predict_proba(X_test_xgb)[:, 1]
auc_roc_lgbm = roc_auc_score(y_test, y_prob_lgbm)
auc_pr_lgbm  = average_precision_score(y_test, y_prob_lgbm)

print(f"LightGBM (default)  —  árvores usadas: {lgbm.best_iteration_} / {lgbm.n_estimators}")
print(f"  AUC-ROC (teste) : {auc_roc_lgbm:.4f}")
print(f"  AUC-PR  (teste) : {auc_pr_lgbm:.4f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators = 300,
    max_depth    = None,
    class_weight = "balanced",
    random_state = 42,
    n_jobs       = -1,
)
rf.fit(X_train_xgb, y_train)

y_prob_rf  = rf.predict_proba(X_test_xgb)[:, 1]
auc_roc_rf = roc_auc_score(y_test, y_prob_rf)
auc_pr_rf  = average_precision_score(y_test, y_prob_rf)

print(f"Random Forest (default)")
print(f"  AUC-ROC (teste) : {auc_roc_rf:.4f}")
print(f"  AUC-PR  (teste) : {auc_pr_rf:.4f}")

In [ ]:
# Tabela final da Seção 6 — escolher o vencedor para o Optuna
resultados = pd.DataFrame([
    {"Modelo": "DummyClassifier",    "AUC-ROC": auc_roc_dummy, "AUC-PR": auc_pr_dummy},
    {"Modelo": "LogisticRegression", "AUC-ROC": auc_roc_lr,    "AUC-PR": auc_pr_lr},
    {"Modelo": "XGBoost",            "AUC-ROC": auc_roc_xgb,   "AUC-PR": auc_pr_xgb},
    {"Modelo": "LightGBM",           "AUC-ROC": auc_roc_lgbm,  "AUC-PR": auc_pr_lgbm},
    {"Modelo": "Random Forest",      "AUC-ROC": auc_roc_rf,    "AUC-PR": auc_pr_rf},
]).sort_values("AUC-ROC", ascending=False).reset_index(drop=True)

print("=== Comparação final — modelos default ===\n")
print(resultados.to_string(index=False, float_format="{:.4f}".format))

vencedor = resultados.iloc[0]["Modelo"]
print(f"\n→ Vencedor: {vencedor}  →  segue para tuning no Optuna (Seção 8)")